<a href="https://colab.research.google.com/github/naimish75/AI-Powered-Adverse-Event-Forecasting-Using-Temporal-Data/blob/main/Stream_Lit_(Drug).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')


Mounted at /content/gdrive


In [ ]:
!pip install streamlit -q
!pip install unidecode
!pip install OpenAI
!python -m pip install "numpy<2"
!pip install pmdarima==2.0.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 65.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!wget -q -O - https://loca.lt/mytunnelpassword

34.139.255.202

In [ ]:
%%writefile app.py
import openai
import base64
import streamlit as st
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
from transformers import pipeline
from unidecode import unidecode
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error
from datetime import timedelta
import io
from PIL import Image
from sklearn.preprocessing import MinMaxScaler

# ===========================
# 📦 Utility Loaders
# ===========================

@st.cache_data
def load_interaction_data():
    with open("/content/gdrive/MyDrive/Drug_interaction_data/Drug_Info.Json", "r") as f:
        return json.load(f)

@st.cache_resource
def load_summarizer():
    return pipeline("summarization", model="facebook/bart-large-cnn")

@st.cache_data
def load_faers_data():
    df = pd.read_csv("/content/gdrive/MyDrive/Drug_interaction_data/aggregated_drug_data_Drugs.csv")
    return df

def normalize(text):
    return unidecode(text.strip().upper())

# ===========================
# 🔎 Interaction Logic
# ===========================

def find_drug_by_name(name, interaction_data):
    name = normalize(name)
    for entry in interaction_data:
        drug = entry.get("drug", {})
        if normalize(drug.get("medicine_name", "")) == name:
            return drug
    return None

def analyze_interactions(existing_meds, new_med, interaction_data):
    interactions = []
    new_drug = find_drug_by_name(new_med, interaction_data)
    if not new_drug:
        st.warning(f"New medication '{new_med}' not found in Drug_Info.Json.")
        return interactions

    for med in existing_meds:
        if med == new_med:
            continue
        existing_drug = find_drug_by_name(med, interaction_data)
        if not existing_drug:
            st.warning(f"Existing medication '{med}' not found in Drug_Info.Json.")
            continue

        for ingr1 in existing_drug.get("active_ingredients", []):
            quantity1 = ingr1.get("quantity", {})
            if quantity1.get("value") == "N/A" or quantity1.get("unit") == "N/A":
                continue

            for ingr2 in new_drug.get("active_ingredients", []):
                quantity2 = ingr2.get("quantity", {})
                if quantity2.get("value") == "N/A" or quantity2.get("unit") == "N/A":
                    continue

                if normalize(ingr1["name"]) == normalize(ingr2["name"]):
                    interactions.append({
                        "Existing Medicine": med,
                        "New Medicine": new_med,
                        "Ingredient": ingr1["name"].upper(),
                        "Interaction": f"Potential overlap of {ingr1['name'].upper()}",
                        "Severity": "Moderate"
                    })
    return interactions

def ask_llm_about_ingredient_types(ingredients_list):
    prompt = f"""
You are a medical assistant.

Given the following overlapping ingredients, for each ingredient indicate whether it is ACTIVE or INACTIVE.

Based on your judgment:
- If most ingredients are INACTIVE, mention clearly that severity can be considered low.
- If most are ACTIVE, assess normally.
- If a mix, assess cautiously.

Here are the ingredients:
{ingredients_list}

Respond in 3-5 lines maximum. Be clear.
"""

    try:
        response = client.chat.completions.create(
            model="gpt-4-turbo-2024-04-09",
            messages=[
                {"role": "system", "content": "You are a helpful medical assistant."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=300
        )
        output = response.choices[0].message.content
        return output
    except Exception as e:
        st.error(f"Error asking GPT-4 about ingredient types: {e}")
        return None


def summarize_interactions(interactions, summarizer):
    summaries = []
    for i in interactions:
        text = f"Interaction between {i['Existing Medicine']} and {i['New Medicine']}: {i['Interaction']}. Severity: {i['Severity']}."

        # Dynamically decide max_length
        input_length = len(text.split())
        target_max_length = max(15, int(input_length * 0.6))  # output 60% length

        summary = summarizer(text, max_length=target_max_length, min_length=10, do_sample=False)
        summaries.append({
            "Drugs": f"{i['Existing Medicine']} + {i['New Medicine']}",
            "Summary": summary[0]['summary_text']
        })
    return summaries




def normalize_forecast(forecast_df):
    norm_df = []
    for drug in forecast_df["drug"].unique():
        df = forecast_df[forecast_df["drug"] == drug].copy()
        scaler = MinMaxScaler()
        df["forecast"] = scaler.fit_transform(df[["forecast"]])
        norm_df.append(df)
    return pd.concat(norm_df)


def plot_combined_forecast(forecast_df):
    fig, ax = plt.subplots(figsize=(12, 6))

    for drug in forecast_df["drug"].unique():
        drug_df = forecast_df[forecast_df["drug"] == drug]

        # Skip if drug forecast is flat (all 0 or very close to 0)
        if np.allclose(drug_df["forecast"], 0):
            continue

        ax.plot(
            drug_df["ds"],
            drug_df["forecast"],
            label=f"{drug} (Forecast)",
            linestyle='--'
        )

    ax.set_title("Future Adverse Reaction Forecast (Next 3 Months)")
    ax.set_ylabel("Forecasted Adverse Reaction Risk")
    ax.set_xlabel("Month")
    ax.legend(title="Drugs", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45)
    plt.tight_layout()

    return fig


def summarize_graph_image(fig, meds):
    # Save plot to in-memory image
    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    buf.seek(0)
    image = Image.open(buf)

    # Call your vision LLM here
    # Replace with GeminiVision / BLIP / Hugging Face API
    prompt = f"This graph shows adverse event predictions for the following medications over the next 1 months: {', '.join(meds)}. Summarize the long-term impact if these medications are continued together."

    vision_summary = simple_vision_llm(prompt=prompt, image=image)
    return vision_summary

def simple_vision_llm(prompt, image):
    return f"[Mock LLM Insight]: If {', '.join(prompt.split(':')[1].split(','))} are taken long-term, monitoring is advised based on forecasted uptick."

# ===========================
# 📈 Forecasting Logic
# ===========================

def forecast_trend_for_drugs(drug_list, faers_data):
    combined_data = []
    for drug_name in drug_list:
        drug_name = normalize(drug_name)
        filtered = faers_data[faers_data["drug"] == drug_name]

        if filtered.empty or len(filtered) < 12:
            continue

        faers_data['report_date'] = pd.to_datetime(faers_data['report_date'])
        filtered['report_date'] = pd.to_datetime(filtered['report_date'])

        trend = filtered.set_index("report_date").resample("M").count().reset_index()
        trend = trend[["report_date", "reaction"]].rename(columns={"report_date": "ds", "reaction": "y"})

        if trend.empty or len(trend) < 6:
            continue  # skip if too little data

        full_train_data = trend

        try:
            model = auto_arima(
                full_train_data["y"],
                seasonal=True,
                m=12,
                suppress_warnings=True,
                error_action="ignore",
                stepwise=True
            )
        except ValueError:
            model = auto_arima(
                full_train_data["y"],
                seasonal=False,
                suppress_warnings=True,
                error_action="ignore"
            )

        # ✅ Forecast next 3 months into future
        n_future_months = 3
        future_forecast = model.predict(n_periods=n_future_months)

        # ✅ Create proper monthly future dates
        last_date = full_train_data['ds'].max()
        future_dates = pd.date_range(last_date + pd.DateOffset(months=1), periods=n_future_months, freq='MS')

        df = pd.DataFrame({
            "drug": drug_name,
            "ds": future_dates,
            "actual": [np.nan] * n_future_months,
            "forecast": future_forecast
        })

        combined_data.append(df)

    return pd.concat(combined_data) if combined_data else pd.DataFrame()


# ===========================
# 🚀 LLM INTEGRATION
# ===========================



# ✅ Set your API key once (or inside the function if preferred)
client = openai.OpenAI(api_key="YOUR_API_KEY")  # replace with your actual key

def generate_gpt4_summary(interaction_text, fig, drug_list):
    # Convert the matplotlib figure to base64 image string
    buf = io.BytesIO()
    fig.savefig(buf, format="png")
    buf.seek(0)
    encoded_image = base64.b64encode(buf.read()).decode()

    prompt = f"""
    You are a helpful medical assistant.
    Give me a concise summary containing majorly only and only two points which are the interaction summary and the graph Explanation.\n
    Graph Explanation should be more about the future like how adverse the reaction can get over time.
    It should have no Title.|
    Here is a summary of drug interactions:
    {interaction_text}

    Below is a graph showing forecasted adverse reactions for these medications: {', '.join(drug_list)}.

    Using both the interaction summary and the graph, provide a clear and concise medical insight about risks or considerations when taking these medications together.
    """

    # 🧠 New Chat API with Vision (openai>=1.0.0)
    response = client.chat.completions.create(
        model="gpt-4-turbo-2024-04-09",
        messages=[
            {"role": "system", "content": "You are a helpful medical assistant."},
            {"role": "user", "content": [
                {"type": "text", "text": prompt},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{encoded_image}"
                    }
                }
            ]}
        ],
        max_tokens=300
    )

    return response.choices[0].message.content

def ask_llm_for_severity(interaction_list):
    prompt = f"""
    You are a medical assistant.

    Assign a severity level (Mild, Moderate, Severe, Critical) to each of the following drug interactions based on medical risk. Return a JSON list where each entry has 'drugs' and 'severity'.

    Interactions:
    {interaction_list}
      """

    try:
        response = client.chat.completions.create(
            model="gpt-4-turbo-2024-04-09",
            messages=[
                {"role": "system", "content": "You are a helpful medical assistant."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=500
        )
        output = response.choices[0].message.content
        return output  # We will parse this later
    except Exception as e:
        st.error(f"Error asking GPT-4 for severity: {e}")
        return None



# ===========================
# 🚀 Main Streamlit App
# ===========================

def main():
    st.title("💊 Drug Interaction + Future Risk Forecasting")
    interaction_data = load_interaction_data()
    summarizer = load_summarizer()
    faers_data = load_faers_data()

    st.write("### Enter your medications below:")

    existing_meds = [normalize(med) for med in st.text_area("Current Medications (one per line):").splitlines() if med.strip()]
    new_med = normalize(st.text_input("New Medication to Add:"))

    if st.button("Check Interactions & Predict Risk"):
        if not existing_meds or not new_med:
            st.error("Please enter both current and new medication.")
            return

        # ✅ Step 1: Interactions
        st.subheader("🔍 Interaction Check")
        interactions = analyze_interactions(existing_meds, new_med, interaction_data)
        if interactions:
          st.dataframe(pd.DataFrame(interactions))
          st.subheader("🧠 Summarized Insights")

          for summary in summarize_interactions(interactions, summarizer):
              st.markdown(f"**{summary['Drugs']}**")
              st.write(summary["Summary"])

          # 🧠 New Step: Ingredient-based Severity Assessment
          ingredients_list_for_llm = "\n".join([
              f"{i['Ingredient']} - {i['Type']}"
              for i in interactions
          ])

          ingredient_severity_advice = ask_llm_about_ingredient_types(ingredients_list_for_llm)

          if ingredient_severity_advice:
              st.subheader("💬 Ingredient Type Risk Analysis")
              st.info(ingredient_severity_advice)

          # Then normally proceed to Severity Prediction
          interaction_text_for_llm = "\n".join([
              f"{i['Existing Medicine']} + {i['New Medicine']}: {i['Interaction']}"
              for i in interactions
          ])

          severity_output = ask_llm_for_severity(interaction_text_for_llm)

          if severity_output:
            st.subheader("🩺 Predicted Severity by GPT-4")
            st.code(severity_output, language="json")
          else:
            st.success("No significant interactions found.")



        all_meds = existing_meds + [new_med]
        valid_meds = [drug for drug in all_meds if find_drug_by_name(drug, interaction_data)]

        # 🔧 FIXED: Call forecasting only once for all valid meds
        forecast_df = forecast_trend_for_drugs(valid_meds, faers_data)

        if forecast_df.empty:
            st.warning("Not enough data available in FAERS to generate a forecast.")
            return

        # 📊 Plot combined forecast
        norm_forecast_df = normalize_forecast(forecast_df)
        fig = plot_combined_forecast(norm_forecast_df)

        if interactions:
          interaction_text = "\n".join([
          f"- {i['Existing Medicine']} + {i['New Medicine']}: {i['Interaction']} (Severity: {i['Severity']})"
          for i in interactions
        ])
        else:
            interaction_text = "No known direct interactions found."

        # 🧠 GPT-4 Combined Summary
        # st.subheader("🧠 Combined Risk Summary")
        try:
            combined_summary = generate_gpt4_summary(
                interaction_text=interaction_text,
                fig=fig,
                drug_list=list(forecast_df['drug'].unique())
            )
            st.markdown(
            f"""
            <div style="background-color:transparent !important; padding:15px; border-radius:10px; border:1px solid #ddd;">
                {combined_summary}
            </div>
            """,
            unsafe_allow_html=True
        )

            st.pyplot(fig)
        except Exception as e:
            st.error(f"Error generating summary from GPT-4: {e}")
if __name__ == "__main__":
    main()


Writing app.py


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501 #34.139.255.202



⠙⠹⠸
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.139.255.202:8501

⠼⠴Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼your url is: https://icy-crabs-walk.loca.lt
2025-04-29 18:30:34.602782: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745951434.631078    1103 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745951434.640237    1103 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-29 18:30:34.670111: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is 